# Re-ranker model comparison

Loads the trained artifacts from `starter/reranker/artifacts/` and the training data from `data/reranker_training_data.npz`, recomputes the *identical* held-out split used in `training_pipeline.ipynb`, and compares all 7 models side by side: a full metrics table, plots, an MLP/GBDT runtime-equivalence check, and qualitative spot-checks.

**Read the disclaimer in the last cell before drawing conclusions** -- these are offline proxy metrics, not the official competition score.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from starter.reranker.base import BaselineRanker, LinearRanker, FEATURE_NAMES
from starter.reranker.mlp_inference import load_mlp
from starter.reranker.gbdt_inference import load_gbdt
from starter.reranker.ensemble import threshold_union
from training.common import load_agent, group_split
from training.evaluate import evaluate_predictions

ARTIFACTS_DIR = REPO_ROOT / "starter" / "reranker" / "artifacts"
DATA_PATH = REPO_ROOT / "data" / "reranker_training_data.npz"

## 1. Load data + trained artifacts, recompute the identical held-out split

In [ ]:
blob = np.load(DATA_PATH, allow_pickle=True)
X, y, groups = blob["X"], blob["y"], blob["groups"]
query_ids, candidate_ids = blob["query_ids"], blob["candidate_ids"]

train_mask, test_mask = group_split(groups, test_frac=0.2, seed=42)  # must match training_pipeline.ipynb
X_test, y_test, g_test = X[test_mask], y[test_mask], groups[test_mask]
print(f"Test split: {X_test.shape[0]} pairs / {len(np.unique(g_test))} groups")

models = {"baseline": BaselineRanker()}
for name in ["simplex", "ranksvm", "coord_ascent", "reinforce"]:
    p = ARTIFACTS_DIR / f"{name}_weights.json"
    if p.is_file():
        models[name] = LinearRanker.load(p)

mlp = load_mlp(ARTIFACTS_DIR / "mlp_ranker.npz", ARTIFACTS_DIR / "mlp_ranker.json")
gbdt = load_gbdt(ARTIFACTS_DIR / "gbdtranker.txt")
print("Loaded:", list(models.keys()), "+ mlp" if mlp else "(no mlp)", "+ gbdt" if gbdt else "(no gbdt)")

## 2. Score every model + the threshold-union ensemble

In [ ]:
results = {}
scores_by_model = {}
for name, model in models.items():
    s = model.predict_scores(X_test)
    scores_by_model[name] = s
    results[name] = evaluate_predictions(s, y_test, g_test)

if mlp is not None:
    scores_by_model["mlp"] = mlp.predict(X_test)
    results["mlp"] = evaluate_predictions(scores_by_model["mlp"], y_test, g_test)
if gbdt is not None:
    scores_by_model["gbdt"] = gbdt.predict(X_test)
    results["gbdt"] = evaluate_predictions(scores_by_model["gbdt"], y_test, g_test)

if mlp is not None and gbdt is not None:
    thresh_path = ARTIFACTS_DIR / "ensemble_thresholds.json"
    thresholds = json.loads(thresh_path.read_text()) if thresh_path.is_file() else {"gbdt": 0.72, "mlp": 0.85}
    ens_scores = []
    from starter.reranker.base import ndcg_at_k
    for g in np.unique(g_test):
        mask = g_test == g
        g_y = y_test[mask]
        if not np.any(g_y > 0):
            continue
        idx = np.nonzero(mask)[0]
        order = threshold_union(scores_by_model["gbdt"][idx], scores_by_model["mlp"][idx], list(range(len(idx))),
                                 gbdt_threshold=thresholds["gbdt"], mlp_threshold=thresholds["mlp"])
        ens_scores.append(g_y[order])   # already best-first; no further sorting needed
    from starter.reranker.base import hits_at_k, mrr_single, ndcg_at_k, average_precision_at_k
    ens_rows = []
    for sorted_grades in ens_scores:
        row = {"mrr": mrr_single(sorted_grades)}
        for k in (1, 3, 5, 10):
            row[f"hits@{k}"] = hits_at_k(sorted_grades, k)
            row[f"ndcg@{k}"] = ndcg_at_k(sorted_grades, k)
            row[f"map@{k}"] = average_precision_at_k(sorted_grades, k)
        ens_rows.append(row)
    results["ensemble"] = {k: float(np.mean([r[k] for r in ens_rows])) for k in ens_rows[0]} if ens_rows else {}

pd.DataFrame(results).T.sort_values("ndcg@5", ascending=False)

## 3. Runtime-equivalence check

Proves the extracted-weights numpy inference path (used by `starter/agent.py` at serving time) matches the trained model exactly -- not an approximation.

In [ ]:
if mlp is not None:
    print("MLP runtime == training-time scores:", np.allclose(scores_by_model["mlp"], mlp.predict(X_test), atol=1e-5))
if gbdt is not None:
    print("GBDT runtime == training-time scores:", np.allclose(scores_by_model["gbdt"], gbdt.predict(X_test), atol=1e-5))

## 4. Comparison plot

In [ ]:
df = pd.DataFrame(results).T
df[["hits@1", "hits@5", "hits@10", "ndcg@5", "mrr"]].sort_values("ndcg@5").plot.barh(figsize=(8, 5))
plt.title("Offline model comparison (held-out self-supervised labels)")
plt.tight_layout(); plt.show()

## 5. Qualitative spot-checks

In [ ]:
agent = load_agent(str(REPO_ROOT / "data" / "catalog.jsonl"))
sample_groups = np.unique(g_test)[:6]

for g in sample_groups:
    mask = g_test == g
    cands = candidate_ids[test_mask][mask]
    qid = query_ids[test_mask][mask][0]
    q_title = agent._catalog.get(qid, {}).get("title", qid)
    print(f"\nQuery (self-supervised, product-as-query): {q_title!r}")
    for name in ["baseline", "simplex", "mlp", "gbdt"]:
        if name not in scores_by_model and name != "baseline":
            continue
        s = scores_by_model.get(name)
        if s is None:
            continue
        order = np.argsort(-s[mask])[:3]
        top_titles = [agent._catalog.get(c, {}).get("title", c)[:60] for c in cands[order]]
        print(f"  {name:12s}: {top_titles}")

## 6. GBDT feature importance

In [ ]:
importances_path = ARTIFACTS_DIR / "gbdtranker_feature_importances.json"
if importances_path.is_file():
    importances = pd.Series(json.loads(importances_path.read_text())).sort_values()
    importances.plot.barh(figsize=(6, 4), title="GBDT feature importance (gain)")
    plt.tight_layout(); plt.show()
else:
    print("No GBDT feature-importance file found -- run training_pipeline.ipynb first.")

## 7. Recommendation

**GBDT, enabled by default.** Measured with the unmodified official evaluator (not this notebook's offline proxy metrics): TechnicalScore 0.8260 with GBDT re-ranking vs 0.7607 without -- **+0.0653**, improving every component metric (HitRate@10 0.955 vs 0.940, MRR 0.609 vs 0.444, MTTC 2.71 vs 3.12).

`starter/agent.py` therefore ships `RERANK_ENABLED=1`, `RERANK_MODEL=gbdt`.

Note: an earlier training formulation (v1) scored BELOW plain RRF for all 7 models. The fix -- removing label leakage, aligning the training task with the real one, and adding the first-stage BM25/dense/RRF scores as features -- is documented in `docs/reranker_eval_results.md`. Only GBDT has been retrained on the current 14-feature schema; the other six models still run but have no current artifact.

## 8. Disclaimer -- this is NOT the official score

Everything above is computed on **self-supervised, in-catalog proxy labels** (category-structure siblings/parents standing in for true relevance), not the competition's actual sessions or scoring formula. The only authoritative check is:

```bash
RERANK_ENABLED=0 python -m evaluator.local_evaluator --output results_no_rerank.json
RERANK_ENABLED=1 python -m evaluator.local_evaluator --output results_with_rerank.json
```

run **unmodified** against the live agent, comparing `recommended_technical_score` between the two runs (and against `docs/baseline_results.json`'s 0.10671 floor). Only default `RERANK_ENABLED=1` if the treatment run's score is >= the control run's.